# Model Evaluation

This notebook evaluates the trained Temporal Fusion Transformer (TFT)
on the final 60-day test period.

The TFT uses 120 days of historical context to forecast up to 60 future days.

We evaluate forecast quality at multiple horizons:
- 7 days
- 14 days
- 30 days
- 60 days

We then compare TFT against:
1. a seasonal baseline
2. a stronger LightGBM forecasting baseline

Evaluation uses MAE and WAPE, followed later by rolling backtesting.

In [2]:
# Pandas is used for loading and working with tabular data.
import pandas as pd

# NumPy is useful for numerical calculations.
import numpy as np

# PyTorch is used because TFT predictions are returned as tensors.
import torch

# PyTorch Forecasting provides the dataset structure and TFT model.
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

In [5]:
# Load the model-ready time-series data created in Notebook 03.
data = pd.read_csv(
    "../data/favorita/store3_item589403_tft_ready.csv"
)

# Convert the date column from text into a datetime type.
data["date"] = pd.to_datetime(data["date"])

# Check that the data loaded correctly.
print("Rows:", len(data))
print(
    "Date range:",
    data["date"].min(),
    "to",
    data["date"].max()
)

Rows: 1687
Date range: 2013-01-02 00:00:00 to 2017-08-15 00:00:00


In [6]:
# Use the same forecasting architecture as the final TFT in Notebook 04.
#
# TFT sees up to 120 historical days and predicts up to 60 future days.
max_encoder_length = 120
max_prediction_length = 60


# Reserve the final 60 days as the untouched test period.
test_length = max_prediction_length

# Reserve the 60 days immediately before test for validation.
validation_length = max_prediction_length


# Final 60 days -> test set.
test_df = data.iloc[
    -test_length:
].copy()


# 60 days immediately before test -> validation set.
validation_df = data.iloc[
    -(test_length + validation_length):-test_length
].copy()


# Everything before validation + test -> training set.
train_df = data.iloc[
    :-(test_length + validation_length)
].copy()


# Verify that evaluation uses the same split as Notebook 04.
print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print(
    "\nTest date range:",
    test_df["date"].min(),
    "to",
    test_df["date"].max()
)

Training rows: 1567
Validation rows: 60
Test rows: 60

Test date range: 2017-06-17 00:00:00 to 2017-08-15 00:00:00


In [9]:
# These columns must be categorical strings,
# matching the setup used during model training.
categorical_columns = [
    "series_id",
    "day_of_week",
    "month",
    "promotion_status"
]

for column in categorical_columns:
    data[column] = data[column].astype(str)


# Recreate the same variable groups used in Notebook 04.
static_categoricals = [
    "series_id"
]

time_varying_known_categoricals = [
    "day_of_week",
    "month",
    "promotion_status"
]

time_varying_known_reals = [
    "time_idx",
    "year",
    "day_of_month"
]

time_varying_unknown_reals = [
    "unit_sales"
]


# The final training time index.
training_cutoff = train_df["time_idx"].max()


# Recreate the training dataset configuration.
#
# We are NOT retraining here.
# This object is needed so the test dataset uses
# exactly the same feature setup as the trained TFT.
training_dataset = TimeSeriesDataSet(
    data[data["time_idx"] <= training_cutoff],

    time_idx="time_idx",
    target="unit_sales",
    group_ids=["series_id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=static_categoricals,
    time_varying_known_categoricals=time_varying_known_categoricals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
)


# Create the final test forecasting window.
#
# Because predict=True, this selects the final available
# 60-day history -> 14-day forecast sample.
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data,
    predict=True,
    stop_randomization=True
)


# Convert the test dataset into a DataLoader.
test_dataloader = test_dataset.to_dataloader(
    train=False,
    batch_size=64,
    shuffle=False
)


print("Test samples:", len(test_dataset))
print("Test batches:", len(test_dataloader))

Test samples: 1
Test batches: 1


In [7]:
# Path to the final TFT model trained in Notebook 04.
#
# This checkpoint uses:
# 120 historical days -> 60 future days.
checkpoint_path = (
    "lightning_logs/version_6/checkpoints/best-tft-60day.ckpt"
)


# Load the trained 60-day TFT model from disk.
best_tft = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint_path
)

print("60-day TFT model loaded successfully.")

60-day TFT model loaded successfully.


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [10]:
# Generate forecasts for the final 60-day test period.
#
# mode="prediction" returns one predicted sales value
# for each day in the forecast horizon.
test_predictions = best_tft.predict(
    test_dataloader,
    mode="prediction"
)

# Check that TFT produced exactly 60 future predictions.
print("Prediction shape:", test_predictions.shape)
print(test_predictions)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint

Prediction shape: torch.Size([1, 60])
tensor([[13.6772, 12.4999,  6.8989,  6.9270,  6.1280,  6.1684, 15.2026, 13.3129,
         12.0683,  6.8845,  6.3712,  6.1404,  5.9363, 14.8874, 15.8786, 13.8347,
          8.1037, 10.6419,  6.7620,  7.1958, 16.1850, 15.5264, 13.5811,  7.9059,
          9.5297,  6.6950,  6.9569, 16.0544, 14.9593, 13.2159,  7.6698,  8.2231,
          6.6157,  6.6030, 15.8291, 14.1691, 12.7205,  7.4226,  7.1143,  6.5173,
          6.2304, 15.5269, 13.3302, 12.3397,  7.2974, 10.4833,  6.2967,  7.2443,
         16.0429, 15.9915, 13.7429,  7.6644, 11.2033,  6.2451,  7.0991, 15.8883,
         15.5007, 13.4495,  7.3713,  9.7261]])


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


In [11]:
# We have one time series, so [0] selects its complete
# 60-day TFT forecast.
#
# detach() removes the tensor from PyTorch's gradient graph,
# cpu() ensures it is on the CPU,
# and numpy() converts it into a NumPy array.
predicted_sales = (
    test_predictions[0]
    .detach()
    .cpu()
    .numpy()
)


# Get the actual sales for the same 60-day test period.
actual_sales = test_df["unit_sales"].to_numpy()


# Combine dates, actual sales, and TFT predictions
# so that each forecast can be compared with what actually happened.
comparison_df = pd.DataFrame({
    "date": test_df["date"].to_numpy(),
    "actual_sales": actual_sales,
    "tft_prediction": predicted_sales
})


# Safety check: both should contain exactly 60 values.
print("Number of TFT predictions:", len(predicted_sales))
print("Number of actual values:", len(actual_sales))


comparison_df.head(10)

Number of TFT predictions: 60
Number of actual values: 60


,date,actual_sales,tft_prediction
0,2017-06-17,22.0,13.677161
1,2017-06-18,9.0,12.499903
2,2017-06-19,5.0,6.898939
3,2017-06-20,7.0,6.926966
4,2017-06-21,3.0,6.127982
5,2017-06-22,5.0,6.168433
6,2017-06-23,7.0,15.202622
7,2017-06-24,9.0,13.312930
8,2017-06-25,5.0,12.068273
9,2017-06-26,7.0,6.884463


In [12]:
# Calculate the absolute TFT forecasting error for every test day.
#
# Example:
# actual = 10, prediction = 7
# absolute error = 3
comparison_df["tft_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["tft_prediction"]
).abs()


# We want to know how TFT performs not just over all 60 days,
# but also at shorter forecasting horizons.
forecast_horizons = [7, 14, 30, 60]


# We will store the metric results for each horizon here.
tft_metric_rows = []


for horizon in forecast_horizons:

    # Take only the first N days of the 60-day forecast.
    #
    # For example:
    # horizon = 7  -> evaluate days 1-7
    # horizon = 30 -> evaluate days 1-30
    horizon_df = comparison_df.iloc[:horizon]


    # MAE = average absolute forecasting error.
    mae = horizon_df["tft_absolute_error"].mean()


    # Total forecasting error across this horizon.
    total_absolute_error = (
        horizon_df["tft_absolute_error"].sum()
    )


    # Total actual demand across the same horizon.
    total_actual_sales = (
        horizon_df["actual_sales"].sum()
    )


    # WAPE = total absolute error relative to total actual demand.
    #
    # The safety check prevents division by zero if a period
    # happens to contain zero total sales.
    if total_actual_sales != 0:
        wape = (
            total_absolute_error
            / total_actual_sales
        ) * 100
    else:
        wape = np.nan


    # Store the results for this horizon.
    tft_metric_rows.append({
        "horizon_days": horizon,
        "MAE": mae,
        "WAPE_percent": wape
    })


# Convert all horizon results into a clean table.
tft_metrics_df = pd.DataFrame(tft_metric_rows)

tft_metrics_df

,horizon_days,MAE,WAPE_percent
0,7,3.756250,45.334055
1,14,3.384534,45.561031
2,30,4.165460,57.586998
3,60,4.040435,57.311134


## 7-Day Seasonal Baseline

The seasonal baseline predicts each day's demand using the actual sales
observed seven days earlier.

This provides a simple benchmark for determining whether the TFT learns
useful patterns beyond basic weekly seasonality.

In [16]:
# Create a leakage-free seasonal baseline using only
# ACTUALLY OBSERVED sales from the original Favorita series.

# Keep only observations that occurred before the test period begins.
raw_pre_test = raw_series[
    raw_series["date"] < test_df["date"].min()
].copy()


# Add the weekday for each genuinely observed sales date.
raw_pre_test["day_of_week"] = (
    raw_pre_test["date"].dt.day_name()
)


# For each weekday, take the MOST RECENT genuinely observed
# sales value available before the test period.
#
# Example:
# future Monday -> latest observed Monday sales before test
# future Tuesday -> latest observed Tuesday sales before test
last_observed_by_weekday = (
    raw_pre_test
    .sort_values("date")
    .groupby("day_of_week")
    .tail(1)
    .set_index("day_of_week")["unit_sales"]
    .to_dict()
)


# Create one seasonal prediction for every test date.
seasonal_predictions = []

for date in test_df["date"]:

    weekday = date.day_name()

    seasonal_predictions.append(
        last_observed_by_weekday[weekday]
    )


print("Number of seasonal predictions:", len(seasonal_predictions))
print("Latest observed sales by weekday:")
print(last_observed_by_weekday)

Number of seasonal predictions: 60
Latest observed sales by weekday:
{'Saturday': 18.0, 'Sunday': 37.0, 'Monday': 8.0, 'Tuesday': 4.0, 'Wednesday': 1.0, 'Thursday': 1.0, 'Friday': 2.0}


In [14]:
# Inspect the final 14 days that were actually known
# before the 60-day test period begins.
#
# This helps us verify why our most recent 7-day pattern
# contains so many zero-sales days.
pre_test_history[
    [
        "date",
        "unit_sales",
        "promotion_status"
    ]
].tail(14)

,date,unit_sales,promotion_status
1613,2017-06-03,0.0,unknown
1614,2017-06-04,0.0,unknown
1615,2017-06-05,0.0,unknown
1616,2017-06-06,0.0,unknown
1617,2017-06-07,0.0,unknown
1618,2017-06-08,0.0,unknown
1619,2017-06-09,0.0,unknown
1620,2017-06-10,0.0,unknown
1621,2017-06-11,0.0,unknown
1622,2017-06-12,0.0,unknown


In [18]:
# Load the original observed Store 3 / Item 589403 series.
#
# We keep this separate from the continuous daily dataset because
# missing calendar dates in the prepared TFT data were filled with 0.
# For the seasonal baseline, we want to use only genuinely observed sales.
raw_series = pd.read_csv(
    "../data/favorita/store3_item589403.csv"
)

# Convert the date column back into datetime format.
raw_series["date"] = pd.to_datetime(
    raw_series["date"]
)

print("Observed historical rows:", len(raw_series))

Observed historical rows: 1593


In [17]:
# Add the seasonal baseline forecasts to our comparison table.
comparison_df["seasonal_prediction"] = seasonal_predictions


# Calculate the absolute error for each seasonal prediction.
comparison_df["seasonal_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["seasonal_prediction"]
).abs()


# Evaluate the seasonal baseline at the same horizons as TFT.
seasonal_metric_rows = []


for horizon in forecast_horizons:

    # Use only the first N days for this horizon.
    horizon_df = comparison_df.iloc[:horizon]


    # Mean Absolute Error.
    seasonal_mae = (
        horizon_df["seasonal_absolute_error"].mean()
    )


    # Total absolute error across this horizon.
    seasonal_total_absolute_error = (
        horizon_df["seasonal_absolute_error"].sum()
    )


    # Total actual demand across the same horizon.
    seasonal_total_actual_sales = (
        horizon_df["actual_sales"].sum()
    )


    # Calculate WAPE safely.
    if seasonal_total_actual_sales != 0:
        seasonal_wape = (
            seasonal_total_absolute_error
            / seasonal_total_actual_sales
        ) * 100
    else:
        seasonal_wape = np.nan


    # Save the result for this horizon.
    seasonal_metric_rows.append({
        "horizon_days": horizon,
        "MAE": seasonal_mae,
        "WAPE_percent": seasonal_wape
    })


# Convert the results into a clean table.
seasonal_metrics_df = pd.DataFrame(
    seasonal_metric_rows
)

seasonal_metrics_df

,horizon_days,MAE,WAPE_percent
0,7,7.000000,84.482759
1,14,7.714286,103.846154
2,30,8.400000,116.129032
3,60,8.266667,117.257683


## LightGBM Forecasting Baseline

LightGBM does not directly process a sequence of historical sales like TFT.
Instead, we create tabular features that describe recent demand behavior,
including lag values, rolling averages, calendar information, and
promotion status.

In [19]:
# Create a separate copy of the prepared dataset for LightGBM.
# We use a copy so that we do not accidentally modify the original `data`
# dataframe that we are already using for TFT evaluation.
lightgbm_df = data.copy()


# Create lag features from historical sales.
#
# A lag feature tells LightGBM:
# "What were sales N days ago?"
#
# Example:
# lag_7 = sales exactly one week earlier.

# Sales from 1 day earlier.
lightgbm_df["lag_1"] = lightgbm_df["unit_sales"].shift(1)

# Sales from 7 days earlier.
lightgbm_df["lag_7"] = lightgbm_df["unit_sales"].shift(7)

# Sales from 14 days earlier.
lightgbm_df["lag_14"] = lightgbm_df["unit_sales"].shift(14)

# Sales from 28 days earlier.
# This gives the model roughly one month of seasonal context.
lightgbm_df["lag_28"] = lightgbm_df["unit_sales"].shift(28)

# Sales from 56 days earlier.
# Since our maximum forecast horizon is 60 days,
# this gives LightGBM information from roughly eight weeks earlier.
lightgbm_df["lag_56"] = lightgbm_df["unit_sales"].shift(56)


# Display a small portion of the dataframe so we can inspect
# how the lag values line up with the original sales.
lightgbm_df[
    [
        "date",
        "unit_sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ]
].head(65)

,date,unit_sales,lag_1,lag_7,lag_14,lag_28,lag_56
0,2013-01-02,1.0,NaN,NaN,NaN,NaN,NaN
1,2013-01-03,3.0,1.0,NaN,NaN,NaN,NaN
2,2013-01-04,3.0,3.0,NaN,NaN,NaN,NaN
3,2013-01-05,3.0,3.0,NaN,NaN,NaN,NaN
4,2013-01-06,2.0,3.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
60,2013-03-03,5.0,3.0,5.0,5.0,6.0,2.0
61,2013-03-04,2.0,5.0,3.0,5.0,2.0,5.0
62,2013-03-05,8.0,2.0,4.0,11.0,7.0,6.0
63,2013-03-06,2.0,8.0,3.0,6.0,2.0,3.0


In [20]:
# Create rolling-average features from historical sales.
#
# First shift sales by 1 day so today's actual sales are NOT
# accidentally used to predict today's target.
historical_sales = lightgbm_df["unit_sales"].shift(1)


# Average sales over the previous 7 days.
lightgbm_df["rolling_mean_7"] = (
    historical_sales
    .rolling(window=7)
    .mean()
)


# Average sales over the previous 14 days.
lightgbm_df["rolling_mean_14"] = (
    historical_sales
    .rolling(window=14)
    .mean()
)


# Average sales over the previous 28 days.
lightgbm_df["rolling_mean_28"] = (
    historical_sales
    .rolling(window=28)
    .mean()
)


# Average sales over the previous 56 days.
#
# This gives LightGBM a smoother view of roughly
# two months of recent demand behavior.
lightgbm_df["rolling_mean_56"] = (
    historical_sales
    .rolling(window=56)
    .mean()
)


# Inspect when the rolling features become available.
lightgbm_df[
    [
        "date",
        "unit_sales",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_mean_56"
    ]
].head(65)

,date,unit_sales,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_56
0,2013-01-02,1.0,NaN,NaN,NaN,NaN
1,2013-01-03,3.0,NaN,NaN,NaN,NaN
2,2013-01-04,3.0,NaN,NaN,NaN,NaN
3,2013-01-05,3.0,NaN,NaN,NaN,NaN
4,2013-01-06,2.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
60,2013-03-03,5.0,5.142857,6.071429,5.785714,4.857143
61,2013-03-04,2.0,5.142857,6.071429,5.750000,4.910714
62,2013-03-05,8.0,5.000000,5.857143,5.750000,4.857143
63,2013-03-06,2.0,5.571429,5.642857,5.785714,4.892857


In [ ]:
# Display all column names currently available in our LightGBM dataframe.
# This lets us confirm which calendar and promotion features already exist.
print(lightgbm_df.columns.tolist())

In [21]:
# Define the columns that LightGBM will use to predict demand.
feature_columns = [
    # Historical demand features
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",

    # Rolling demand features
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",

    # Long-term time/trend information
    "time_idx",

    # Calendar features
    "day_of_week",
    "month",

    # Promotion information
    "promotion_status",
]

# The target is what LightGBM is trying to predict.
target_column = "unit_sales"

print("LightGBM features:")
print(feature_columns)

print("\nTarget:")
print(target_column)

LightGBM features:
['lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_56', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_56', 'time_idx', 'day_of_week', 'month', 'promotion_status']

Target:
unit_sales


In [22]:
# Remove rows where lag or rolling features are not yet available.
#
# Because lag_56 and rolling_mean_56 need 56 previous days,
# the earliest rows cannot be used for LightGBM training.
lightgbm_model_df = lightgbm_df.dropna(
    subset=feature_columns + [target_column]
).copy()


print("Original rows:", len(lightgbm_df))
print("Rows available for LightGBM:", len(lightgbm_model_df))
print("Rows removed because history was insufficient:",
      len(lightgbm_df) - len(lightgbm_model_df))


# Show the first usable LightGBM row.
lightgbm_model_df[
    ["date"] + feature_columns + [target_column]
].head()

Original rows: 1687
Rows available for LightGBM: 1631
Rows removed because history was insufficient: 56


,date,lag_1,lag_7,lag_14,lag_28,lag_56,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_56,time_idx,day_of_week,month,promotion_status,unit_sales
56,2013-02-27,4.0,6.0,3.0,3.0,1.0,5.714286,6.000000,5.571429,4.607143,56,Wednesday,2,unknown,3.0
57,2013-02-28,3.0,8.0,9.0,4.0,3.0,5.285714,6.000000,5.571429,4.642857,57,Thursday,2,unknown,11.0
58,2013-03-01,11.0,6.0,5.0,8.0,3.0,5.714286,6.142857,5.821429,4.785714,58,Friday,3,unknown,7.0
59,2013-03-02,7.0,8.0,6.0,3.0,3.0,5.857143,6.285714,5.785714,4.857143,59,Saturday,3,unknown,3.0
60,2013-03-03,3.0,5.0,5.0,6.0,2.0,5.142857,6.071429,5.785714,4.857143,60,Sunday,3,unknown,5.0


In [25]:
# Match the LightGBM time split to the TFT setup.

train_end_date = pd.Timestamp("2017-04-17")
validation_start_date = pd.Timestamp("2017-04-18")
validation_end_date = pd.Timestamp("2017-06-16")
test_start_date = pd.Timestamp("2017-06-17")


# Training rows
lightgbm_train_df = lightgbm_model_df[
    lightgbm_model_df["date"] <= train_end_date
].copy()


# Validation rows
lightgbm_validation_df = lightgbm_model_df[
    (lightgbm_model_df["date"] >= validation_start_date)
    & (lightgbm_model_df["date"] <= validation_end_date)
].copy()


# Test rows
lightgbm_test_df = lightgbm_model_df[
    lightgbm_model_df["date"] >= test_start_date
].copy()


print("LightGBM training rows:", len(lightgbm_train_df))
print("LightGBM validation rows:", len(lightgbm_validation_df))
print("LightGBM test rows:", len(lightgbm_test_df))

print(
    "\nTraining period:",
    lightgbm_train_df["date"].min(),
    "to",
    lightgbm_train_df["date"].max()
)

print(
    "Validation period:",
    lightgbm_validation_df["date"].min(),
    "to",
    lightgbm_validation_df["date"].max()
)

print(
    "Test period:",
    lightgbm_test_df["date"].min(),
    "to",
    lightgbm_test_df["date"].max()
)

LightGBM training rows: 1511
LightGBM validation rows: 60
LightGBM test rows: 60

Training period: 2013-02-27 00:00:00 to 2017-04-17 00:00:00
Validation period: 2017-04-18 00:00:00 to 2017-06-16 00:00:00
Test period: 2017-06-17 00:00:00 to 2017-08-15 00:00:00


In [26]:
# -------------------------
# Training features/target
# -------------------------

X_train = lightgbm_train_df[
    feature_columns
].copy()

y_train = lightgbm_train_df[
    target_column
].copy()


# -------------------------
# Validation features/target
# -------------------------

X_validation = lightgbm_validation_df[
    feature_columns
].copy()

y_validation = lightgbm_validation_df[
    target_column
].copy()


# Tell pandas/LightGBM which columns are categorical.
categorical_columns = [
    "day_of_week",
    "month",
    "promotion_status",
]


for column in categorical_columns:
    X_train[column] = X_train[column].astype("category")
    X_validation[column] = X_validation[column].astype("category")


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation.shape)

X_train shape: (1511, 13)
y_train shape: (1511,)
X_validation shape: (60, 13)
y_validation shape: (60,)


In [28]:
# Import the LightGBM regressor and early-stopping callback.
from lightgbm import LGBMRegressor, early_stopping


# Create the LightGBM regression model.
#
# We deliberately allow many trees here because early stopping
# will stop training once validation performance stops improving.
lightgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)


# Train the model using the training set.
#
# The validation set is NOT used to directly fit the trees.
# It is used to monitor whether performance is still improving.
lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_validation, y_validation)
    ],
    eval_metric="mae",
    callbacks=[
        early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)


print("\nLightGBM training completed.")
print("Best iteration:", lightgbm_model.best_iteration_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Number of data points in the train set: 1511, number of used features: 13
[LightGBM] [Info] Start training from score 5.838518
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[82]	valid_0's l1: 5.62757	valid_0's l2: 57.581

LightGBM training completed.
Best iteration: 82


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [29]:
# We will build the 60-day forecast one day at a time.
#
# Important:
# once we enter the test period, we must NOT use the real future
# sales values to build lag/rolling features.
# We will use our own previous predictions instead.

# Start with all real sales available before the test period.
history_values = (
    lightgbm_model_df[
        lightgbm_model_df["date"] < test_start_date
    ]["unit_sales"]
    .tolist()
)

# Store each future LightGBM prediction here.
lightgbm_predictions = []


print("Historical values available before forecast:", len(history_values))
print("Forecast days to generate:", len(lightgbm_test_df))

Historical values available before forecast: 1571
Forecast days to generate: 60


In [30]:
# Generate the 60-day LightGBM forecast recursively.
#
# For each future day:
# 1. Build lag features from the history we currently know.
# 2. Build rolling-average features from that same history.
# 3. Add calendar/promotion information for the future day.
# 4. Ask LightGBM for a prediction.
# 5. Add that prediction to history so it can be used
#    when forecasting later days.

for _, future_row in lightgbm_test_df.iterrows():

    # -------------------------
    # Historical lag features
    # -------------------------
    lag_1 = history_values[-1]
    lag_7 = history_values[-7]
    lag_14 = history_values[-14]
    lag_28 = history_values[-28]
    lag_56 = history_values[-56]


    # -------------------------
    # Rolling-average features
    # -------------------------
    rolling_mean_7 = np.mean(history_values[-7:])
    rolling_mean_14 = np.mean(history_values[-14:])
    rolling_mean_28 = np.mean(history_values[-28:])
    rolling_mean_56 = np.mean(history_values[-56:])


    # Build one row containing exactly the same
    # 13 features used during LightGBM training.
    prediction_row = pd.DataFrame({
        "lag_1": [lag_1],
        "lag_7": [lag_7],
        "lag_14": [lag_14],
        "lag_28": [lag_28],
        "lag_56": [lag_56],

        "rolling_mean_7": [rolling_mean_7],
        "rolling_mean_14": [rolling_mean_14],
        "rolling_mean_28": [rolling_mean_28],
        "rolling_mean_56": [rolling_mean_56],

        # These values are known for the future date.
        "time_idx": [future_row["time_idx"]],
        "day_of_week": [future_row["day_of_week"]],
        "month": [future_row["month"]],
        "promotion_status": [future_row["promotion_status"]],
    })


    # Match the categorical types/categories used during training.
    for column in categorical_columns:
        prediction_row[column] = pd.Categorical(
            prediction_row[column],
            categories=X_train[column].cat.categories
        )


    # Make sure columns appear in exactly the same order
    # as they did during training.
    prediction_row = prediction_row[feature_columns]


    # Predict sales for this future day.
    prediction = lightgbm_model.predict(
        prediction_row,
        num_iteration=lightgbm_model.best_iteration_
    )[0]


    # Sales demand cannot physically be negative.
    # If LightGBM predicts below zero, clip it to zero.
    prediction = max(0.0, prediction)


    # Save this day's forecast.
    lightgbm_predictions.append(prediction)


    # IMPORTANT:
    # Add the prediction — NOT the actual future sales —
    # to history for subsequent forecast days.
    history_values.append(prediction)


print("LightGBM predictions generated:", len(lightgbm_predictions))
print("First 10 predictions:")
print(lightgbm_predictions[:10])

LightGBM predictions generated: 60
First 10 predictions:
[np.float64(8.718418815566775), np.float64(8.534090980032339), np.float64(5.927276784312721), np.float64(5.802627889756326), np.float64(5.909917330892914), np.float64(6.014920862715672), np.float64(13.033533169598606), np.float64(7.395323114334347), np.float64(6.590246702388291), np.float64(5.806436816652588)]


In [31]:
# Add the LightGBM predictions to the same comparison dataframe
# that already contains actual sales and TFT predictions.

comparison_df["lightgbm_prediction"] = lightgbm_predictions


# Check the first few rows.
comparison_df[
    [
        "date",
        "actual_sales",
        "tft_prediction",
        "lightgbm_prediction"
    ]
].head(10)

,date,actual_sales,tft_prediction,lightgbm_prediction
0,2017-06-17,22.0,13.677161,8.718419
1,2017-06-18,9.0,12.499903,8.534091
2,2017-06-19,5.0,6.898939,5.927277
3,2017-06-20,7.0,6.926966,5.802628
4,2017-06-21,3.0,6.127982,5.909917
5,2017-06-22,5.0,6.168433,6.014921
6,2017-06-23,7.0,15.202622,13.033533
7,2017-06-24,9.0,13.312930,7.395323
8,2017-06-25,5.0,12.068273,6.590247
9,2017-06-26,7.0,6.884463,5.806437


In [32]:
# Calculate absolute errors for both forecasting models.
comparison_df["tft_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["tft_prediction"]
).abs()

comparison_df["lightgbm_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["lightgbm_prediction"]
).abs()


# Evaluate both models at the same forecast horizons.
forecast_horizons = [7, 14, 30, 60]

comparison_metric_rows = []


for horizon in forecast_horizons:

    # Take only the first N forecast days.
    horizon_df = comparison_df.iloc[:horizon]

    # -------------------------
    # TFT metrics
    # -------------------------
    tft_mae = horizon_df[
        "tft_absolute_error"
    ].mean()

    tft_wape = (
        horizon_df["tft_absolute_error"].sum()
        / horizon_df["actual_sales"].sum()
    ) * 100


    # -------------------------
    # LightGBM metrics
    # -------------------------
    lightgbm_mae = horizon_df[
        "lightgbm_absolute_error"
    ].mean()

    lightgbm_wape = (
        horizon_df["lightgbm_absolute_error"].sum()
        / horizon_df["actual_sales"].sum()
    ) * 100


    # Store the results.
    comparison_metric_rows.append({
        "horizon_days": horizon,

        "TFT_MAE": tft_mae,
        "LightGBM_MAE": lightgbm_mae,

        "TFT_WAPE_percent": tft_wape,
        "LightGBM_WAPE_percent": lightgbm_wape
    })


# Convert the results into a clean comparison table.
model_comparison_df = pd.DataFrame(
    comparison_metric_rows
)

model_comparison_df

,horizon_days,TFT_MAE,LightGBM_MAE,TFT_WAPE_percent,LightGBM_WAPE_percent
0,7,3.756250,3.690073,45.334055,44.535363
1,14,3.384534,2.800566,45.561031,37.699921
2,30,4.165460,3.229621,57.586998,44.649134
3,60,4.040435,2.834312,57.311134,40.203004


# Model Evaluation

This notebook evaluates the trained Temporal Fusion Transformer (TFT)
on the final 60-day test period.

The TFT uses 120 days of historical context to forecast up to 60 future days.

We evaluate forecast quality at multiple horizons:
- 7 days
- 14 days
- 30 days
- 60 days

We then compare TFT against:
1. a seasonal baseline
2. a stronger LightGBM forecasting baseline

Evaluation uses MAE and WAPE, followed later by rolling backtesting.

In [ ]:
# Pandas is used for loading and working with tabular data.
import pandas as pd

# NumPy is useful for numerical calculations.
import numpy as np

# PyTorch is used because TFT predictions are returned as tensors.
import torch

# PyTorch Forecasting provides the dataset structure and TFT model.
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

In [ ]:
# Use the same forecasting architecture as the final TFT in Notebook 04.
#
# TFT sees up to 120 historical days and predicts up to 60 future days.
max_encoder_length = 120
max_prediction_length = 60


# Reserve the final 60 days as the untouched test period.
test_length = max_prediction_length

# Reserve the 60 days immediately before test for validation.
validation_length = max_prediction_length


# Final 60 days -> test set.
test_df = data.iloc[
    -test_length:
].copy()


# 60 days immediately before test -> validation set.
validation_df = data.iloc[
    -(test_length + validation_length):-test_length
].copy()


# Everything before validation + test -> training set.
train_df = data.iloc[
    :-(test_length + validation_length)
].copy()


# Verify that evaluation uses the same split as Notebook 04.
print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print(
    "\nTest date range:",
    test_df["date"].min(),
    "to",
    test_df["date"].max()
)

Training rows: 1567
Validation rows: 60
Test rows: 60

Test date range: 2017-06-17 00:00:00 to 2017-08-15 00:00:00


In [ ]:
# These columns must be categorical strings,
# matching the setup used during model training.
categorical_columns = [
    "series_id",
    "day_of_week",
    "month",
    "promotion_status"
]

for column in categorical_columns:
    data[column] = data[column].astype(str)


# Recreate the same variable groups used in Notebook 04.
static_categoricals = [
    "series_id"
]

time_varying_known_categoricals = [
    "day_of_week",
    "month",
    "promotion_status"
]

time_varying_known_reals = [
    "time_idx",
    "year",
    "day_of_month"
]

time_varying_unknown_reals = [
    "unit_sales"
]


# The final training time index.
training_cutoff = train_df["time_idx"].max()


# Recreate the training dataset configuration.
#
# We are NOT retraining here.
# This object is needed so the test dataset uses
# exactly the same feature setup as the trained TFT.
training_dataset = TimeSeriesDataSet(
    data[data["time_idx"] <= training_cutoff],

    time_idx="time_idx",
    target="unit_sales",
    group_ids=["series_id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=static_categoricals,
    time_varying_known_categoricals=time_varying_known_categoricals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
)


# Create the final test forecasting window.
#
# Because predict=True, this selects the final available
# 60-day history -> 14-day forecast sample.
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data,
    predict=True,
    stop_randomization=True
)


# Convert the test dataset into a DataLoader.
test_dataloader = test_dataset.to_dataloader(
    train=False,
    batch_size=64,
    shuffle=False
)


print("Test samples:", len(test_dataset))
print("Test batches:", len(test_dataloader))

Test samples: 1
Test batches: 1


In [ ]:
# Path to the final TFT model trained in Notebook 04.
#
# This checkpoint uses:
# 120 historical days -> 60 future days.
checkpoint_path = (
    "lightning_logs/version_6/checkpoints/best-tft-60day.ckpt"
)


# Load the trained 60-day TFT model from disk.
best_tft = TemporalFusionTransformer.load_from_checkpoint(
    checkpoint_path
)

print("60-day TFT model loaded successfully.")

60-day TFT model loaded successfully.


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [ ]:
# Generate forecasts for the final 60-day test period.
#
# mode="prediction" returns one predicted sales value
# for each day in the forecast horizon.
test_predictions = best_tft.predict(
    test_dataloader,
    mode="prediction"
)

# Check that TFT produced exactly 60 future predictions.
print("Prediction shape:", test_predictions.shape)
print(test_predictions)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint

Prediction shape: torch.Size([1, 60])
tensor([[13.6772, 12.4999,  6.8989,  6.9270,  6.1280,  6.1684, 15.2026, 13.3129,
         12.0683,  6.8845,  6.3712,  6.1404,  5.9363, 14.8874, 15.8786, 13.8347,
          8.1037, 10.6419,  6.7620,  7.1958, 16.1850, 15.5264, 13.5811,  7.9059,
          9.5297,  6.6950,  6.9569, 16.0544, 14.9593, 13.2159,  7.6698,  8.2231,
          6.6157,  6.6030, 15.8291, 14.1691, 12.7205,  7.4226,  7.1143,  6.5173,
          6.2304, 15.5269, 13.3302, 12.3397,  7.2974, 10.4833,  6.2967,  7.2443,
         16.0429, 15.9915, 13.7429,  7.6644, 11.2033,  6.2451,  7.0991, 15.8883,
         15.5007, 13.4495,  7.3713,  9.7261]])


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


In [ ]:
# We have one time series, so [0] selects its complete
# 60-day TFT forecast.
#
# detach() removes the tensor from PyTorch's gradient graph,
# cpu() ensures it is on the CPU,
# and numpy() converts it into a NumPy array.
predicted_sales = (
    test_predictions[0]
    .detach()
    .cpu()
    .numpy()
)


# Get the actual sales for the same 60-day test period.
actual_sales = test_df["unit_sales"].to_numpy()


# Combine dates, actual sales, and TFT predictions
# so that each forecast can be compared with what actually happened.
comparison_df = pd.DataFrame({
    "date": test_df["date"].to_numpy(),
    "actual_sales": actual_sales,
    "tft_prediction": predicted_sales
})


# Safety check: both should contain exactly 60 values.
print("Number of TFT predictions:", len(predicted_sales))
print("Number of actual values:", len(actual_sales))


comparison_df.head(10)

Number of TFT predictions: 60
Number of actual values: 60


,date,actual_sales,tft_prediction
0,2017-06-17,22.0,13.677161
1,2017-06-18,9.0,12.499903
2,2017-06-19,5.0,6.898939
3,2017-06-20,7.0,6.926966
4,2017-06-21,3.0,6.127982
5,2017-06-22,5.0,6.168433
6,2017-06-23,7.0,15.202622
7,2017-06-24,9.0,13.312930
8,2017-06-25,5.0,12.068273
9,2017-06-26,7.0,6.884463


In [ ]:
# Calculate the absolute TFT forecasting error for every test day.
#
# Example:
# actual = 10, prediction = 7
# absolute error = 3
comparison_df["tft_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["tft_prediction"]
).abs()


# We want to know how TFT performs not just over all 60 days,
# but also at shorter forecasting horizons.
forecast_horizons = [7, 14, 30, 60]


# We will store the metric results for each horizon here.
tft_metric_rows = []


for horizon in forecast_horizons:

    # Take only the first N days of the 60-day forecast.
    #
    # For example:
    # horizon = 7  -> evaluate days 1-7
    # horizon = 30 -> evaluate days 1-30
    horizon_df = comparison_df.iloc[:horizon]


    # MAE = average absolute forecasting error.
    mae = horizon_df["tft_absolute_error"].mean()


    # Total forecasting error across this horizon.
    total_absolute_error = (
        horizon_df["tft_absolute_error"].sum()
    )


    # Total actual demand across the same horizon.
    total_actual_sales = (
        horizon_df["actual_sales"].sum()
    )


    # WAPE = total absolute error relative to total actual demand.
    #
    # The safety check prevents division by zero if a period
    # happens to contain zero total sales.
    if total_actual_sales != 0:
        wape = (
            total_absolute_error
            / total_actual_sales
        ) * 100
    else:
        wape = np.nan


    # Store the results for this horizon.
    tft_metric_rows.append({
        "horizon_days": horizon,
        "MAE": mae,
        "WAPE_percent": wape
    })


# Convert all horizon results into a clean table.
tft_metrics_df = pd.DataFrame(tft_metric_rows)

tft_metrics_df

,horizon_days,MAE,WAPE_percent
0,7,3.756250,45.334055
1,14,3.384534,45.561031
2,30,4.165460,57.586998
3,60,4.040435,57.311134


## 7-Day Seasonal Baseline

The seasonal baseline predicts each day's demand using the actual sales
observed seven days earlier.

This provides a simple benchmark for determining whether the TFT learns
useful patterns beyond basic weekly seasonality.

In [ ]:
# Create a leakage-free seasonal baseline using only
# ACTUALLY OBSERVED sales from the original Favorita series.

# Keep only observations that occurred before the test period begins.
raw_pre_test = raw_series[
    raw_series["date"] < test_df["date"].min()
].copy()


# Add the weekday for each genuinely observed sales date.
raw_pre_test["day_of_week"] = (
    raw_pre_test["date"].dt.day_name()
)


# For each weekday, take the MOST RECENT genuinely observed
# sales value available before the test period.
#
# Example:
# future Monday -> latest observed Monday sales before test
# future Tuesday -> latest observed Tuesday sales before test
last_observed_by_weekday = (
    raw_pre_test
    .sort_values("date")
    .groupby("day_of_week")
    .tail(1)
    .set_index("day_of_week")["unit_sales"]
    .to_dict()
)


# Create one seasonal prediction for every test date.
seasonal_predictions = []

for date in test_df["date"]:

    weekday = date.day_name()

    seasonal_predictions.append(
        last_observed_by_weekday[weekday]
    )


print("Number of seasonal predictions:", len(seasonal_predictions))
print("Latest observed sales by weekday:")
print(last_observed_by_weekday)

Number of seasonal predictions: 60
Latest observed sales by weekday:
{'Saturday': 18.0, 'Sunday': 37.0, 'Monday': 8.0, 'Tuesday': 4.0, 'Wednesday': 1.0, 'Thursday': 1.0, 'Friday': 2.0}


In [ ]:
# Add the seasonal baseline forecasts to our comparison table.
comparison_df["seasonal_prediction"] = seasonal_predictions


# Calculate the absolute error for each seasonal prediction.
comparison_df["seasonal_absolute_error"] = (
    comparison_df["actual_sales"]
    - comparison_df["seasonal_prediction"]
).abs()


# Evaluate the seasonal baseline at the same horizons as TFT.
seasonal_metric_rows = []


for horizon in forecast_horizons:

    # Use only the first N days for this horizon.
    horizon_df = comparison_df.iloc[:horizon]


    # Mean Absolute Error.
    seasonal_mae = (
        horizon_df["seasonal_absolute_error"].mean()
    )


    # Total absolute error across this horizon.
    seasonal_total_absolute_error = (
        horizon_df["seasonal_absolute_error"].sum()
    )


    # Total actual demand across the same horizon.
    seasonal_total_actual_sales = (
        horizon_df["actual_sales"].sum()
    )


    # Calculate WAPE safely.
    if seasonal_total_actual_sales != 0:
        seasonal_wape = (
            seasonal_total_absolute_error
            / seasonal_total_actual_sales
        ) * 100
    else:
        seasonal_wape = np.nan


    # Save the result for this horizon.
    seasonal_metric_rows.append({
        "horizon_days": horizon,
        "MAE": seasonal_mae,
        "WAPE_percent": seasonal_wape
    })


# Convert the results into a clean table.
seasonal_metrics_df = pd.DataFrame(
    seasonal_metric_rows
)

seasonal_metrics_df

,horizon_days,MAE,WAPE_percent
0,7,7.000000,84.482759
1,14,7.714286,103.846154
2,30,8.400000,116.129032
3,60,8.266667,117.257683


## LightGBM Forecasting Baseline

LightGBM does not directly process a sequence of historical sales like TFT.
Instead, we create tabular features that describe recent demand behavior,
including lag values, rolling averages, calendar information, and
promotion status.

## LightGBM Forecasting Baseline

LightGBM does not directly read a sequence of historical sales like TFT.
therefore create features that describe recent demand behavior.

We will begin with lag features, which represent sales from previous days.

In [ ]:
# Create a separate copy of the prepared dataset for LightGBM.
# We use a copy so that we do not accidentally modify the original `data`
# dataframe that we are already using for TFT evaluation.
lightgbm_df = data.copy()


# Create lag features from historical sales.
#
# A lag feature tells LightGBM:
# "What were sales N days ago?"
#
# Example:
# lag_7 = sales exactly one week earlier.

# Sales from 1 day earlier.
lightgbm_df["lag_1"] = lightgbm_df["unit_sales"].shift(1)

# Sales from 7 days earlier.
lightgbm_df["lag_7"] = lightgbm_df["unit_sales"].shift(7)

# Sales from 14 days earlier.
lightgbm_df["lag_14"] = lightgbm_df["unit_sales"].shift(14)

# Sales from 28 days earlier.
# This gives the model roughly one month of seasonal context.
lightgbm_df["lag_28"] = lightgbm_df["unit_sales"].shift(28)

# Sales from 56 days earlier.
# Since our maximum forecast horizon is 60 days,
# this gives LightGBM information from roughly eight weeks earlier.
lightgbm_df["lag_56"] = lightgbm_df["unit_sales"].shift(56)


# Display a small portion of the dataframe so we can inspect
# how the lag values line up with the original sales.
lightgbm_df[
    [
        "date",
        "unit_sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ]
].head(65)

,date,unit_sales,lag_1,lag_7,lag_14,lag_28,lag_56
0,2013-01-02,1.0,NaN,NaN,NaN,NaN,NaN
1,2013-01-03,3.0,1.0,NaN,NaN,NaN,NaN
2,2013-01-04,3.0,3.0,NaN,NaN,NaN,NaN
3,2013-01-05,3.0,3.0,NaN,NaN,NaN,NaN
4,2013-01-06,2.0,3.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
60,2013-03-03,5.0,3.0,5.0,5.0,6.0,2.0
61,2013-03-04,2.0,5.0,3.0,5.0,2.0,5.0
62,2013-03-05,8.0,2.0,4.0,11.0,7.0,6.0
63,2013-03-06,2.0,8.0,3.0,6.0,2.0,3.0


In [ ]:
# Create rolling-average features from historical sales.
#
# First shift sales by 1 day so today's actual sales are NOT
# accidentally used to predict today's target.
historical_sales = lightgbm_df["unit_sales"].shift(1)


# Average sales over the previous 7 days.
lightgbm_df["rolling_mean_7"] = (
    historical_sales
    .rolling(window=7)
    .mean()
)


# Average sales over the previous 14 days.
lightgbm_df["rolling_mean_14"] = (
    historical_sales
    .rolling(window=14)
    .mean()
)


# Average sales over the previous 28 days.
lightgbm_df["rolling_mean_28"] = (
    historical_sales
    .rolling(window=28)
    .mean()
)


# Average sales over the previous 56 days.
#
# This gives LightGBM a smoother view of roughly
# two months of recent demand behavior.
lightgbm_df["rolling_mean_56"] = (
    historical_sales
    .rolling(window=56)
    .mean()
)


# Inspect when the rolling features become available.
lightgbm_df[
    [
        "date",
        "unit_sales",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_mean_56"
    ]
].head(65)

,date,unit_sales,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_56
0,2013-01-02,1.0,NaN,NaN,NaN,NaN
1,2013-01-03,3.0,NaN,NaN,NaN,NaN
2,2013-01-04,3.0,NaN,NaN,NaN,NaN
3,2013-01-05,3.0,NaN,NaN,NaN,NaN
4,2013-01-06,2.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
60,2013-03-03,5.0,5.142857,6.071429,5.785714,4.857143
61,2013-03-04,2.0,5.142857,6.071429,5.750000,4.910714
62,2013-03-05,8.0,5.000000,5.857143,5.750000,4.857143
63,2013-03-06,2.0,5.571429,5.642857,5.785714,4.892857


In [ ]:
# Display all column names currently available in our LightGBM dataframe.
# This lets us confirm which calendar and promotion features already exist.
print(lightgbm_df.columns.tolist())

In [ ]:
# Define the columns that LightGBM will use to predict demand.
feature_columns = [
    # Historical demand features
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",

    # Rolling demand features
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",

    # Long-term time/trend information
    "time_idx",

    # Calendar features
    "day_of_week",
    "month",

    # Promotion information
    "promotion_status",
]

# The target is what LightGBM is trying to predict.
target_column = "unit_sales"

print("LightGBM features:")
print(feature_columns)

print("\nTarget:")
print(target_column)

LightGBM features:
['lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_56', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_56', 'time_idx', 'day_of_week', 'month', 'promotion_status']

Target:
unit_sales


In [ ]:
# Remove rows where lag or rolling features are not yet available.
#
# Because lag_56 and rolling_mean_56 need 56 previous days,
# the earliest rows cannot be used for LightGBM training.
lightgbm_model_df = lightgbm_df.dropna(
    subset=feature_columns + [target_column]
).copy()


print("Original rows:", len(lightgbm_df))
print("Rows available for LightGBM:", len(lightgbm_model_df))
print("Rows removed because history was insufficient:",
      len(lightgbm_df) - len(lightgbm_model_df))


# Show the first usable LightGBM row.
lightgbm_model_df[
    ["date"] + feature_columns + [target_column]
].head()

Original rows: 1687
Rows available for LightGBM: 1631
Rows removed because history was insufficient: 56


,date,lag_1,lag_7,lag_14,lag_28,lag_56,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_56,time_idx,day_of_week,month,promotion_status,unit_sales
56,2013-02-27,4.0,6.0,3.0,3.0,1.0,5.714286,6.000000,5.571429,4.607143,56,Wednesday,2,unknown,3.0
57,2013-02-28,3.0,8.0,9.0,4.0,3.0,5.285714,6.000000,5.571429,4.642857,57,Thursday,2,unknown,11.0
58,2013-03-01,11.0,6.0,5.0,8.0,3.0,5.714286,6.142857,5.821429,4.785714,58,Friday,3,unknown,7.0
59,2013-03-02,7.0,8.0,6.0,3.0,3.0,5.857143,6.285714,5.785714,4.857143,59,Saturday,3,unknown,3.0
60,2013-03-03,3.0,5.0,5.0,6.0,2.0,5.142857,6.071429,5.785714,4.857143,60,Sunday,3,unknown,5.0


In [ ]:
# Match the LightGBM time split to the TFT setup.

train_end_date = pd.Timestamp("2017-04-17")
validation_start_date = pd.Timestamp("2017-04-18")
validation_end_date = pd.Timestamp("2017-06-16")
test_start_date = pd.Timestamp("2017-06-17")


# Training rows
lightgbm_train_df = lightgbm_model_df[
    lightgbm_model_df["date"] <= train_end_date
].copy()


# Validation rows
lightgbm_validation_df = lightgbm_model_df[
    (lightgbm_model_df["date"] >= validation_start_date)
    & (lightgbm_model_df["date"] <= validation_end_date)
].copy()


# Test rows
lightgbm_test_df = lightgbm_model_df[
    lightgbm_model_df["date"] >= test_start_date
].copy()


print("LightGBM training rows:", len(lightgbm_train_df))
print("LightGBM validation rows:", len(lightgbm_validation_df))
print("LightGBM test rows:", len(lightgbm_test_df))

print(
    "\nTraining period:",
    lightgbm_train_df["date"].min(),
    "to",
    lightgbm_train_df["date"].max()
)

print(
    "Validation period:",
    lightgbm_validation_df["date"].min(),
    "to",
    lightgbm_validation_df["date"].max()
)

print(
    "Test period:",
    lightgbm_test_df["date"].min(),
    "to",
    lightgbm_test_df["date"].max()
)

LightGBM training rows: 1511
LightGBM validation rows: 60
LightGBM test rows: 60

Training period: 2013-02-27 00:00:00 to 2017-04-17 00:00:00
Validation period: 2017-04-18 00:00:00 to 2017-06-16 00:00:00
Test period: 2017-06-17 00:00:00 to 2017-08-15 00:00:00


In [ ]:
# -------------------------
# Training features/target
# -------------------------

X_train = lightgbm_train_df[
    feature_columns
].copy()

y_train = lightgbm_train_df[
    target_column
].copy()


# -------------------------
# Validation features/target
# -------------------------

X_validation = lightgbm_validation_df[
    feature_columns
].copy()

y_validation = lightgbm_validation_df[
    target_column
].copy()


# Tell pandas/LightGBM which columns are categorical.
categorical_columns = [
    "day_of_week",
    "month",
    "promotion_status",
]


for column in categorical_columns:
    X_train[column] = X_train[column].astype("category")
    X_validation[column] = X_validation[column].astype("category")


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation.shape)

X_train shape: (1511, 13)
y_train shape: (1511,)
X_validation shape: (60, 13)
y_validation shape: (60,)
